In [ ]:
from pathlib import Path

cwd = Path.cwd()
parent_dir = cwd.parent
print(f"parent dir: {parent_dir}")

In [ ]:
import os
import requests

def save_and_cache(url, path, force=False):
    if not os.path.exists(path) or force:
        print(f"Downloading {url} to {path} ...")
        r = requests.get(url)
        with open(path, 'wb') as f:
            f.write(r.content)

# 移动数据

In [ ]:
import pandas as pd

target_states = {
    'Alaska': {"code": "02", "abbr": "ak"}, # Alaska
    'Illinois': {"code": "17", "abbr": "il"}, # Illinois
    'Maine': {"code": "23", "abbr": "me"}, # Maine
    'Minnesota': {"code": "27", "abbr": "mn"}, # Minnesota
    'Utah': {"code": "49", "abbr": "ut"}, # Utah
    'Washington': {"code": "53", "abbr": "wa"}  # Washington
}


GEOID2Name = {}

print("Building GEOID2Name mapping table...")

for state_name, state in target_states.items():
    # Construct the URL for the official Census FIPS data file
    # Format: https://www2.census.gov/geo/docs/reference/codes/files/st{FIPS}_{ABBR}_cou.txt
    url = f"https://www2.census.gov/geo/docs/reference/codes/files/st{state['code']}_{state['abbr']}_cou.txt"
    
    try:
        # Read data (no header, comma-separated)
        # Column definition: State(AK), StateFP(02), CountyFP(013), CountyName(Aleutians East Borough), ClassFP
        df = pd.read_csv(url, header=None, dtype=str)
        
        # Iterate through each row, populate the dictionary
        for _, row in df.iterrows():
            state_fp = row[1]
            county_fp = row[2]
            county_name = row[3]
            
            # Construct unique FIPS integer ID (StateFP + CountyFP)
            # Example: IL(17) + Cook(031) -> "17031" -> int(17031)
            geoid = int(state_fp + county_fp)
            
            GEOID2Name[geoid] = county_name
            
        print(f"  - Loaded: {state['abbr'].upper()} (containing {len(df)} counties/districts)")
        
    except Exception as e:
        print(f"  ! Error: Unable to load {state['abbr'].upper()} - {e}")

print("Mapping table construction completed.\n")

# --- Test ---
# 17031 is Cook County in Illinois
# 53033 is King County in Washington
print(f"Test: 17031 -> {GEOID2Name.get(17031, 'Not Found')}")
print(f"Test: 53033 -> {GEOID2Name.get(53033, 'Not Found')}")

In [ ]:
import numpy as np
import json
import pandas as pd


def matrix_to_df(matrix, node_names, edge_num=100):
    # 1. Copy the matrix to avoid modifying the original data
    mat = matrix.copy()
    np.fill_diagonal(mat, 0)
    flat_indices = np.argsort(mat.flatten())[-int(edge_num):]
    rows, cols = np.unravel_index(flat_indices, mat.shape)
    
    data_list = []
    for r, c in zip(rows, cols):
        flow = mat[r, c]
        if flow > 0:
            source_name = node_names[r]
            target_name = node_names[c]
            data_list.append({'source': source_name, 'destination': target_name, 'flow': flow})
        
    # 6. Convert to DataFrame and sort by flow from large to small (for subsequent plotting logic)
    df = pd.DataFrame(data_list)
    if not df.empty:
        df = df.sort_values(by='flow', ascending=False).reset_index(drop=True)
        
    return df

covid_path = parent_dir / 'data' / 'covid19' 

covid_Network_Data_path = covid_path / 'COVID_Network_Data'

picked_result_path = parent_dir / 'picked_result' / 'for_covid' 

pred_edges_files = sorted(picked_result_path.glob("*_pred.npy"))

od_dict = {}
pred_dict = {}

for file in pred_edges_files:
    state_anme = file.name.split('_')[0]
    print(f"Loading predictions for {state_anme} from {file.name} ...")
    A_pred = np.load(file)[0]
    A_od = np.load(picked_result_path / 'data' / state_anme / 'A.npy')
    print(f"Loading OD for {state_anme}")
    with open(covid_Network_Data_path / f'COVIDin{state_anme}.json', 'r') as f:
        data = json.load(f)
    print(f"Loading original data for {state_anme}")
    nodes = data["node"]
    print(f'nodes: {len(nodes)}')
    np.fill_diagonal(A_pred, 0)
    np.fill_diagonal(A_od, 0)
    K = int((A_od > 0).sum())
    pred_dict[state_anme] = matrix_to_df(A_pred, nodes, edge_num=K)
    od_dict[state_anme] = matrix_to_df(A_od, nodes, edge_num=K)

    print("===============================")

In [ ]:
import geopandas as gpd

gdf_path = parent_dir / 'data' / 'covid19' 
os.makedirs(gdf_path, exist_ok=True)
gdf_dict = {}

for state_name, state in target_states.items():
    save_and_cache(f'https://www2.census.gov/geo/tiger/TIGER2016/COUSUB/tl_2016_{state["code"]}_cousub.zip',
                f'{gdf_path}/tl_2016_{state["code"]}_cousub.zip')

    gdf = gpd.read_file(f'{gdf_path}/tl_2016_{state["code"]}_cousub.zip')
    gdf_dict[state_name] = gdf
    

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from shapely.ops import unary_union
from scipy.stats import pearsonr
import re

# ==========================================
# 1. String fuzzy matching tool
# ==========================================
def normalize_name(name):
    """
    Normalize place names: convert to lowercase, remove common administrative suffixes, remove punctuation
    """
    name = name.lower().strip()
    # Remove common administrative division suffixes (for US counties)
    suffixes = [
        ' county', ' borough', ' municipality', ' census area', 
        ' city and borough', ' parish', ' city'
    ]
    for suffix in suffixes:
        if name.endswith(suffix):
            name = name.replace(suffix, '')
    
    # Remove punctuation
    name = re.sub(r'[^\w\s]', '', name)
    return name.strip()

def build_location_mapping(json_nodes, gdf):
    """
    Establish mapping from JSON nodes to (x, y) coordinates
    Solve name inconsistency issues (especially for Alaska)
    """
    # 1. Calculate centroids of each polygon in GDF
    gdf_centroids = {} # {Normalized_Name: (x, y)}
    
    for idx, row in gdf.iterrows():
        # Use official name from GEOID2Name or NAME from gdf
        official_name = row['NAME'] 
        norm_name = normalize_name(official_name)
        gdf_centroids[norm_name] = row['geometry'].centroid.coords[0]

    # 2. Try to match JSON Nodes
    node_to_coords = {}
    unmatched = []
    
    for node in json_nodes:
        norm_node = normalize_name(node)
        
        # Strategy A: Exact match (after normalization)
        if norm_node in gdf_centroids:
            node_to_coords[node] = gdf_centroids[norm_node]
            continue
            
        # Strategy B: Substring match (e.g., "Anchorage" in "Municipality of Anchorage")
        match_found = False
        for gdf_name, coords in gdf_centroids.items():
            if norm_node in gdf_name or gdf_name in norm_node:
                node_to_coords[node] = coords
                match_found = True
                break
        
        if match_found:
            continue
            
        unmatched.append(node)
        
    if unmatched:
        print(f"  [Warning] {len(unmatched)} nodes unmatched (e.g., {unmatched[:3]})...")
        
    return node_to_coords

In [ ]:
def plotOD(ax, df, location_dict, vmax=None, alpha=0.8, color_scheme='plasma'):
    """
    Plot OD flow lines, highlighting the top 10% with highest flow in yellow.
    """
    # RASTER_THRESHOLD = 10 
    if df.empty: return

    # 1. Data cleaning
    valid_df = df[df['source'].isin(location_dict) & df['destination'].isin(location_dict)].copy()
    if valid_df.empty: return

    # 2. Sorting and threshold calculation
    # Sort by flow from small to large (for plotting large flows on top)
    valid_df = valid_df.sort_values(by='flow', ascending=True).reset_index(drop=True)
    
    # Calculate cutoff index for top 10%
    total_lines = len(valid_df)
    cutoff_index = int(total_lines * 0.95) # 0.95 means top 95% are normal lines, bottom 5% are highlighted
    
    # 3. Prepare plotting parameters
    if vmax is None:
        vmax = valid_df['flow'].max()
    
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    cmap = plt.get_cmap(color_scheme)
    
    # 4. Loop plotting
    for idx, row in valid_df.iterrows():
        s, d, flow = row['source'], row['destination'], row['flow']
        x1, y1 = location_dict[s]
        x2, y2 = location_dict[d]
        
        # --- Core modification logic ---
        if idx >= cutoff_index:
            # === Top 5% ===
            color = '#f1c40f'  # Bright gold (Flat UI Gold)
            lw = 0.8 + 1.5 * (flow / vmax) 
            
            cur_alpha = 0.95 
            
            z_order = 200 + flow/vmax * 10
            # z_order = 10.1 
        else:
            # === Bottom 90% ===
            color = cmap(norm(flow)) # Use original color scheme (e.g., plasma)
            
            lw = 0.3 + 1.0 * (flow / vmax)
            cur_alpha = max(0.1, min(alpha, 0.3 + 0.6 * (flow / vmax)))
            
            # Lower level
            z_order = 100 + flow/vmax * 10
            # z_order = 2 + (flow/vmax) * 5 

        # Draw Bezier curve
        arrow = patches.FancyArrowPatch(
            (x1, y1), (x2, y2),
            connectionstyle="arc3,rad=0.15",
            color=color,
            linewidth=lw,
            alpha=cur_alpha,
            arrowstyle='-',
            shrinkA=0, shrinkB=0,
            mutation_scale=10,
            zorder=z_order,
        )
        # arrow.set_rasterized(True) 
        ax.add_patch(arrow)
    # ax.set_rasterization_zorder(RASTER_THRESHOLD)

In [ ]:
# Store prepared data for plotting
# Structure: {'Alaska': {'gdf': ..., 'loc_dict': ...}, ...}
processed_maps = {}

print("\nProcessing Geometries and Matching Names...")

for state_name, state_info in target_states.items():
    print(f"--- Processing {state_name} ---")
    
    # 1. Get raw COUSUB GDF
    raw_gdf = gdf_dict[state_name]
    
    # 2. Aggregate: COUSUB -> County
    # Use the GEOID2Name dictionary built earlier
    # Ensure GEOID2Name exists in memory
    tmp_geoms = []
    
    # Group by StateFP and CountyFP
    for (state_fp, county_fp), group in raw_gdf.groupby(['STATEFP', 'COUNTYFP']):
        geoid = int(state_fp + county_fp)
        
        if geoid in GEOID2Name:
            official_name = GEOID2Name[geoid]
            # Geometry union
            poly = unary_union(group['geometry'])
            tmp_geoms.append({'NAME': official_name, 'geometry': poly})
            
    # Create aggregated County level GDF
    county_gdf = gpd.GeoDataFrame(tmp_geoms, crs=raw_gdf.crs)
    
    # 3. Get node names from JSON (for matching)
    # We need to extract from od_dict or pred_dict (assuming they are consistent)
    # Note: source/destination in your code are names
    if state_name in od_dict and not od_dict[state_name].empty:
        # Extract all unique node names from DataFrame
        df_sample = od_dict[state_name]
        json_nodes = list(set(df_sample['source']).union(set(df_sample['destination'])))
    else:
        print(f"  Error: No flow data for {state_name}")
        continue

    # 4. Establish coordinate mapping (handle name inconsistencies)
    loc_dict = build_location_mapping(json_nodes, county_gdf)
    
    processed_maps[state_name] = {
        'gdf': county_gdf,
        'loc_dict': loc_dict
    }

print("Data processing complete.")

In [ ]:
pcc_dict = {'Alaska': [0.8271, 0.82697], 'Illinois': [0.81207, 0.81247], 'Maine': [0.6794, 0.6794], 'Minnesota': [0.72906, 0.72837], 'Utah': [0.77414, 0.77454], 'Washington': [0.7616, 0.76086]}

In [ ]:
# ==========================================
# 3. Plotting loop
# ==========================================
# Increase height slightly to accommodate titles and labels
fig, axes = plt.subplots(nrows=2, ncols=6, figsize=(26, 11), dpi=300)
# wspace left-right spacing, hspace top-bottom spacing
plt.subplots_adjust(wspace=0.1, hspace=0.1)

# Iterate through states in order
sorted_states = sorted(target_states.keys()) 

for col, state_name in enumerate(sorted_states):
    if state_name not in processed_maps:
        continue
        
    # Get data
    map_data = processed_maps[state_name]
    gdf = map_data['gdf']
    loc_dict = map_data['loc_dict']
    
    df_real = od_dict.get(state_name, pd.DataFrame())
    df_pred = pred_dict.get(state_name, pd.DataFrame())
    
    # Calculate Pearson
    # if not df_real.empty and not df_pred.empty:
    #     merged = pd.merge(df_real, df_pred, on=['source', 'destination'], suffixes=('_real', '_pred'))
    #     if len(merged) > 2:
    #         r_val, _ = pearsonr(merged['flow_real'], merged['flow_pred'])
    #     else:
    #         r_val = 0.0
    # else:
    #     r_val = 0.0
    r_val = pcc_dict[state_name][0]
        
    # Calculate local maximum flow (Local Vmax)
    vmax = max(df_real['flow'].max() if not df_real.empty else 0, 
               df_pred['flow'].max() if not df_pred.empty else 0) + 1e-9
    
    # Get subplot objects
    ax_top = axes[0, col]
    ax_bottom = axes[1, col]
    
    # --- Plotting loop ---
    for ax, df, label in zip([ax_top, ax_bottom], [df_real, df_pred], ['GT', 'Pred']):
        # 1. Plot base map
        gdf.plot(ax=ax, color='#f1f2f6', edgecolor='#b2bec3', linewidth=0.3)
        
        # 2. Plot flow (OD)
        plotOD(ax, df, loc_dict, vmax=vmax, alpha=0.7, color_scheme='plasma')
        
        # 3. View adjustment
        ax.axis('off')
        
        if state_name == 'Alaska':
            ax.set_xlim(-175, -130) 
            ax.set_ylim(50, 72)
        else:
            minx, miny, maxx, maxy = gdf.total_bounds
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            ax.set_aspect('equal')

    # ==========================================
    # Core modification: Use blended coordinate system to align text
    # ==========================================
    # trans: x-axis follows subplot (centered), y-axis follows canvas (fixed height)
    trans = mtransforms.blended_transform_factory(ax_top.transAxes, fig.transFigure)
    
    # 1. Title (top aligned)
    # y=0.91: 91% of canvas height
    ax_top.text(0.5, 0.91, f"{state_name}", 
                transform=trans, 
                ha='center', va='bottom', 
                fontsize=20, fontweight='bold', color='#2d3436')
    
    # 2. Pearson label (center aligned)
    # y=0.51: 51% of canvas height (center)
    p_color = '#e17055' if r_val < 0.4 else '#2d3436'
    
    ax_top.text(0.5, 0.51, f"PCC = {r_val:.3f}", 
                transform=trans, 
                ha='center', va='center', 
                fontsize=15, fontweight='bold', color=p_color,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#b2bec3", alpha=1.0, zorder=200))

    # 3. Column separator lines
    # Also use blended coordinates to ensure consistent line height
    if col < 5:
        # Create blended coordinate system
        # X-axis: follows subplot (ax_top.transAxes), ensures line is on the right of each column
        # Y-axis: follows canvas (fig.transFigure), ensures all lines have absolute consistent height
        trans_line = mtransforms.blended_transform_factory(ax_top.transAxes, fig.transFigure)
   
        line = mlines.Line2D([1.05, 1.05], [0.1, 0.9], 
                             transform=trans_line, 
                             color='gray', linewidth=1, linestyle='--', alpha=0.3, clip_on=False)
        ax_top.add_line(line)

# ==========================================
# Row labels (using global coordinates fig.text)
# ==========================================
# x=0.09: leftmost of canvas
# y=0.72: visual center of first row
# y=0.30: visual center of second row
fig.text(0.09, 0.72, "Mobility Graph\n(LODES)", 
         ha='right', va='center', rotation=90, 
         fontsize=18, fontweight='bold', color='#2d3436')

fig.text(0.09, 0.30, "Inferred Graph\n(Model)", 
         ha='right', va='center', rotation=90, 
         fontsize=18, fontweight='bold', color='#2d3436')

save_path = parent_dir / 'data' / 'covid19'

plt.savefig(save_path / 'epidemic_comparison.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
mean_pcc = sum(pcc_dict[state_name][0] for state_name in pcc_dict) / len(pcc_dict)
print(f"Mean PCC: {mean_pcc:.4f}")